# Clase 18 — Random Forest: Muchos Árboles, Mejores Decisiones

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador × UDLA

---

**Objetivos de hoy:**
1. Explorar el dataset Pima Indians Diabetes y entender sus particularidades.
2. Entrenar Logística, Árbol y Random Forest, comparándolos con métricas clínicas.
3. Interpretar cada modelo: coeficientes, reglas, importancias.
4. Entender la diferencia entre **parámetros e hiperparámetros**.
5. Usar **Cross-Validation** y **GridSearchCV** para seleccionar el mejor modelo.
6. Crear una demo interactiva con **Gradio**.

**Dataset:** Pima Indians Diabetes (768 pacientes, 8 variables clínicas).

---

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

---

## 1. Cargar dataset

In [ ]:
d = fetch_openml("diabetes", version=1, as_frame=True, parser="auto")
df = d.frame.copy()
df["target"] = (df["class"] == "tested_positive").astype(int)
df = df.drop(columns=["class"])

print(f"Shape: {df.shape}")
print(df["target"].value_counts(normalize=True))
df.head()

**Variables del dataset:**

| Variable | Significado |
|----------|-------------|
| `preg` | Embarazos |
| `plas` | Glucosa (plasma, 2h OGTT) |
| `pres` | Presión arterial (mm Hg) |
| `skin` | Pliegue cutáneo (mm) |
| `insu` | Insulina (µU/ml) |
| `mass` | BMI (kg/m²) |
| `pedi` | Función pedigrí diabetes |
| `age` | Edad (años) |
| `target` | 1 = diabético, 0 = sano |

---

## 2. EDA — Exploración de datos

In [ ]:
df.describe()

**Noten los ceros sospechosos:** variables como `plas` (glucosa), `pres` (presión), `skin` (pliegue cutáneo), `insu` (insulina) y `mass` (BMI) tienen valores mínimos de **cero**. Pero una glucosa de 0 o un BMI de 0 no es posible fisiológicamente — son **valores faltantes codificados como cero**.

In [ ]:
# Conteo de zeros sospechosos
cols_zeros = ["plas", "pres", "skin", "insu", "mass"]
for col in cols_zeros:
    n_zeros = (df[col] == 0).sum()
    print(f"{col}: {n_zeros} zeros ({n_zeros/len(df)*100:.1f}%)")

In [ ]:
# Reemplazar zeros por NaN e imputar con mediana
for col in cols_zeros:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

print("Zeros después de imputación:")
for col in cols_zeros:
    print(f"  {col}: {(df[col] == 0).sum()}")

In [ ]:
# Histograma de glucosa por clase
fig = px.histogram(
    df, x="plas", color="target", nbins=40, barmode="overlay", opacity=0.7,
    title="Distribución de Glucosa por clase",
    labels={"plas": "Glucosa (plasma)", "count": "Pacientes", "target": "Diabetes"},
    color_discrete_map={0: "#2563EB", 1: "#C82B40"})
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** Pacientes diabéticos tienden a tener glucosa más alta, pero hay solapamiento considerable entre las dos distribuciones. No basta una sola variable para separar perfectamente las clases.

In [ ]:
# Boxplot de todas las variables por clase
numeric_cols = [c for c in df.columns if c != "target"]
df_melt = df.melt(id_vars="target", value_vars=numeric_cols, var_name="variable", value_name="valor")
fig = px.box(df_melt, x="variable", y="valor", color="target",
             title="Distribución de variables por clase",
             color_discrete_map={0: "#2563EB", 1: "#C82B40"})
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** Glucosa (`plas`), BMI (`mass`) y edad (`age`) son las variables que más separan las dos clases. Las medianas y los cuartiles difieren visiblemente entre diabéticos y no diabéticos.

In [ ]:
# Correlación de cada variable con target
correlaciones = df.drop(columns=["target"]).corrwith(df["target"]).sort_values()
fig = px.bar(x=correlaciones.values, y=correlaciones.index, orientation="h",
             title="Correlación de cada variable con diabetes",
             labels={"x": "Correlación", "y": "Variable"},
             color_discrete_sequence=["#2563EB"])
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** Glucosa, BMI, edad y número de embarazos son las variables más correlacionadas con diabetes. Esto confirma lo que vimos en los boxplots.

### Ejercicio 0: EDA completa (10 min)

1. Crea **boxplots individuales** de cada variable numérica coloreados por `target`.
2. ¿Cuáles variables separan mejor las dos clases?
3. ¿Hay alguna variable que NO aporte información para separar diabéticos de sanos?

In [ ]:
# Tu código aquí


---

## 3. Debate de métricas: ¿Qué error es peor?

En un contexto clínico de detección de diabetes:

- **FN (Falso Negativo):** decirle a un diabético que está sano → **peligroso**, no recibe tratamiento.
- **FP (Falso Positivo):** decirle a un sano que puede tener diabetes → se hace más exámenes, costo moderado.

**El FN es mucho peor que el FP.** Por eso:
- Usaremos **AUC** para comparar modelos globalmente.
- Usaremos **Recall** para verificar que detectamos la mayoría de diabéticos.

---

## 4. Preparar datos

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"% positivos train: {y_train.mean()*100:.1f}%")
print(f"% positivos test:  {y_test.mean()*100:.1f}%")

In [ ]:
# Escalar para logística
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

---

## 5. Modelo 1: Regresión Logística

In [ ]:
modelo_lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modelo_lr.fit(X_train_s, y_train)

y_pred_lr = modelo_lr.predict(X_test_s)
y_prob_lr = modelo_lr.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred_lr, target_names=["No Diabetes", "Diabetes"]))
print(f"AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")

In [ ]:
# Matriz de confusión — Logística
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No", "Sí"], yticklabels=["No", "Sí"], ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusión — Logística")
plt.tight_layout()
plt.show()

**Interpretación:** La logística con `class_weight="balanced"` prioriza el recall: detecta más diabéticos a costa de algunos falsos positivos. Revisa los FN (esquina inferior izquierda): cada FN es un diabético que no detectamos.

In [ ]:
# Interpretabilidad — Coeficientes de la logística
coefs = pd.DataFrame({"feature": X.columns, "coef": modelo_lr.coef_[0]}).sort_values("coef")
fig = px.bar(coefs, x="coef", y="feature", orientation="h",
             title="Coeficientes — Regresión Logística",
             color_discrete_sequence=["#2563EB"])
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** Glucosa (`plas`) tiene el coeficiente positivo más grande: mayor glucosa = mayor probabilidad de diabetes. BMI (`mass`) también contribuye positivamente. Los coeficientes negativos (como `pres`) indican que valores más altos reducen ligeramente la probabilidad predicha de diabetes.

---

## 6. Modelo 2: Árbol de Decisión

In [ ]:
arbol = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
arbol.fit(X_train, y_train)

y_pred_arbol = arbol.predict(X_test)
y_prob_arbol = arbol.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_arbol, target_names=["No Diabetes", "Diabetes"]))
print(f"AUC: {roc_auc_score(y_test, y_prob_arbol):.3f}")

In [ ]:
# Matriz de confusión — Árbol
cm_arbol = confusion_matrix(y_test, y_pred_arbol)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_arbol, annot=True, fmt="d", cmap="Reds",
            xticklabels=["No", "Sí"], yticklabels=["No", "Sí"], ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusión — Árbol de Decisión")
plt.tight_layout()
plt.show()

**Interpretación:** Compara los FN del árbol con los de la logística. El árbol puede tener diferente balance precision/recall dependiendo de la estructura aprendida.

In [ ]:
# Interpretabilidad — Visualizar el árbol
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(arbol, ax=ax, filled=True, rounded=True,
          feature_names=list(X.columns),
          class_names=["No Diabetes", "Diabetes"], fontsize=7)
plt.title("Árbol de decisión (max_depth=5)")
plt.tight_layout()
plt.show()

In [ ]:
# Interpretabilidad — Reglas como texto
print(export_text(arbol, feature_names=list(X.columns)))

**Interpretación:** La primera pregunta del árbol es sobre glucosa. Tiene sentido clínico: la glucosa plasmática es el indicador más directo de diabetes. A partir de ahí, el árbol subdivide por BMI, edad y otras variables. Esta estructura es fácil de explicar a un médico.

In [ ]:
# Interpretabilidad — Feature importances del árbol
imp_arbol = pd.DataFrame({"feature": X.columns, "importancia": arbol.feature_importances_}).sort_values("importancia", ascending=True)
fig = px.bar(imp_arbol, x="importancia", y="feature", orientation="h",
             title="Importancia de variables — Árbol de Decisión",
             color_discrete_sequence=["#C82B40"])
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** El árbol concentra la importancia en pocas variables (principalmente glucosa). Esto es típico de árboles individuales: dependen mucho de las primeras divisiones.

---

## 7. Modelo 3: Random Forest

Random Forest entrena **muchos árboles**, cada uno con:
1. **Muestra aleatoria** de los datos (bootstrap / bagging).
2. **Subconjunto aleatorio de features** en cada split.

La predicción final es el **voto de la mayoría**. Al promediar muchos árboles, los errores individuales se cancelan.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                            class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=["No Diabetes", "Diabetes"]))
print(f"AUC: {roc_auc_score(y_test, y_prob_rf):.3f}")
print(f"AP:  {average_precision_score(y_test, y_prob_rf):.3f}")

In [ ]:
# Matriz de confusión — Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens",
            xticklabels=["No", "Sí"], yticklabels=["No", "Sí"], ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusión — Random Forest")
plt.tight_layout()
plt.show()

**Interpretación:** Compara la matriz de RF con las anteriores. RF típicamente logra mejor balance entre precision y recall: detecta más diabéticos (menos FN) sin explotar los falsos positivos.

In [ ]:
# Interpretabilidad — Feature importances: Árbol vs Random Forest (lado a lado)
imp_rf = pd.DataFrame({"feature": X.columns, "importancia": rf.feature_importances_})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Árbol
imp_arbol_sorted = imp_arbol.sort_values("importancia", ascending=True)
axes[0].barh(imp_arbol_sorted["feature"], imp_arbol_sorted["importancia"], color="#C82B40")
axes[0].set_title("Importancia — Árbol de Decisión")
axes[0].set_xlabel("Importancia")

# RF
imp_rf_sorted = imp_rf.sort_values("importancia", ascending=True)
axes[1].barh(imp_rf_sorted["feature"], imp_rf_sorted["importancia"], color="#16A34A")
axes[1].set_title("Importancia — Random Forest")
axes[1].set_xlabel("Importancia")

plt.suptitle("Comparación de Feature Importances", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretación:** RF distribuye la importancia de forma más equitativa entre las variables. Glucosa domina en ambos modelos, pero RF da más peso a insulina, edad y BMI que el árbol individual. Esto ocurre porque cada árbol del bosque usa diferentes subconjuntos de features.

In [ ]:
# Barrido de n_estimators: ¿cuántos árboles necesitamos?
n_trees = [1, 5, 10, 25, 50, 100, 200, 500]
aucs = []
for n in n_trees:
    rf_n = RandomForestClassifier(n_estimators=n, max_depth=8,
                                  class_weight="balanced", random_state=42)
    rf_n.fit(X_train, y_train)
    aucs.append(roc_auc_score(y_test, rf_n.predict_proba(X_test)[:, 1]))

fig = px.line(x=n_trees, y=aucs, markers=True,
              title="AUC vs número de árboles (n_estimators)",
              labels={"x": "n_estimators", "y": "AUC"},
              color_discrete_sequence=["#16A34A"])
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** Después de ~100 árboles la mejora en AUC es marginal. Usar 100-200 árboles es suficiente para este dataset. Más árboles solo aumentan el costo computacional sin beneficio significativo.

---

## 8. Comparación triple: Logística vs Árbol vs Random Forest

In [ ]:
# 3 curvas ROC
fpr_l, tpr_l, _ = roc_curve(y_test, y_prob_lr)
fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_arbol)
fpr_r, tpr_r, _ = roc_curve(y_test, y_prob_rf)

auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_t  = roc_auc_score(y_test, y_prob_arbol)
auc_rf = roc_auc_score(y_test, y_prob_rf)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_l, y=tpr_l, mode="lines",
    name=f"Logística (AUC={auc_lr:.3f})", line=dict(color="#2563EB")))
fig.add_trace(go.Scatter(x=fpr_t, y=tpr_t, mode="lines",
    name=f"Árbol (AUC={auc_t:.3f})", line=dict(color="#C82B40")))
fig.add_trace(go.Scatter(x=fpr_r, y=tpr_r, mode="lines",
    name=f"Random Forest (AUC={auc_rf:.3f})", line=dict(color="#16A34A", width=3)))
fig.add_shape(type="line", line=dict(dash="dash", color="gray"),
              x0=0, x1=1, y0=0, y1=1)
fig.update_layout(title="Curva ROC: Logística vs Árbol vs Random Forest",
    xaxis_title="FPR", yaxis_title="TPR",
    template="plotly_white")
fig.show()

**Interpretación:** RF tiene la curva ROC más alta (más cercana a la esquina superior izquierda), lo que indica mejor capacidad de discriminación en todos los umbrales. La logística también compite bien; el árbol individual es el más débil.

In [ ]:
# 3 curvas Precision-Recall
prec_l, rec_l, _ = precision_recall_curve(y_test, y_prob_lr)
prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_arbol)
prec_r, rec_r, _ = precision_recall_curve(y_test, y_prob_rf)

ap_lr = average_precision_score(y_test, y_prob_lr)
ap_t  = average_precision_score(y_test, y_prob_arbol)
ap_rf = average_precision_score(y_test, y_prob_rf)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rec_l, y=prec_l, mode="lines",
    name=f"Logística (AP={ap_lr:.3f})", line=dict(color="#2563EB")))
fig.add_trace(go.Scatter(x=rec_t, y=prec_t, mode="lines",
    name=f"Árbol (AP={ap_t:.3f})", line=dict(color="#C82B40")))
fig.add_trace(go.Scatter(x=rec_r, y=prec_r, mode="lines",
    name=f"Random Forest (AP={ap_rf:.3f})", line=dict(color="#16A34A", width=3)))
fig.add_hline(y=y_test.mean(), line_dash="dash",
              annotation_text=f"baseline ({y_test.mean():.2%})")
fig.update_layout(title="Curva PR: Logística vs Árbol vs Random Forest",
    xaxis_title="Recall", yaxis_title="Precision",
    template="plotly_white")
fig.show()

**Interpretación:** En la curva PR, la ventaja de RF es aún más clara. La curva PR es especialmente importante en datasets desbalanceados porque no se infla con los verdaderos negativos como la ROC.

### Ejercicio 2: Comparación y decisión clínica (10 min)

1. Imprime el AUC de los 3 modelos.
2. ¿Cuál modelo elegirías para el hospital? ¿Por qué?
3. Si el médico necesita explicar cada caso al paciente, ¿cuál modelo es más fácil de explicar?

In [ ]:
# Tu código aquí


---

## 9. Parámetros vs Hiperparámetros + Cross-Validation

### ¿Cuál es la diferencia?

| | Parámetros | Hiperparámetros |
|---|---|---|
| **¿Quién los ajusta?** | El algoritmo (durante `.fit()`) | El data scientist (antes de `.fit()`) |
| **Ejemplos** | Coeficientes de logística, splits del árbol | `max_depth`, `n_estimators`, `C` |
| **¿Se aprenden de los datos?** | Sí | No, se eligen manualmente o con búsqueda |

### ¿Por qué NO usar el test set para elegir hiperparámetros?

Si probamos muchos hiperparámetros y elegimos el que mejor funciona en test, estamos **"haciendo trampa"**. El test set deja de ser una evaluación honesta — se contamina con nuestras decisiones. Esto se llama **sesgo de selección**.

**Solución: Cross-Validation** — dividimos el train en K partes, entrenamos con K-1 y evaluamos en la restante. Rotamos K veces. Así evaluamos sin tocar el test.

In [ ]:
# Cross-validation simple con RF
rf_cv = RandomForestClassifier(n_estimators=200, max_depth=8,
                               class_weight="balanced", random_state=42)

scores = cross_val_score(rf_cv, X_train, y_train, cv=5, scoring="roc_auc")

print(f"AUC por fold: {scores.round(3)}")
print(f"AUC promedio: {scores.mean():.3f} ± {scores.std():.3f}")

**Interpretación:** El AUC promedio por CV nos da una estimación más robusta del rendimiento real del modelo. La desviación estándar baja indica que el modelo es estable entre diferentes particiones de los datos.

### GridSearchCV — Búsqueda sistemática de hiperparámetros

En vez de probar a mano, `GridSearchCV` prueba **todas las combinaciones** de un grid de hiperparámetros y selecciona la mejor por cross-validation.

In [ ]:
# Definir el grid de hiperparámetros
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 8, 12, None],
    "min_samples_split": [2, 5, 10],
}

print(f"Total de combinaciones: {3 * 4 * 3} = {3*4*3}")

In [ ]:
# GridSearchCV para Random Forest
grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print(f"\nMejor AUC (CV): {grid_rf.best_score_:.3f}")
print(f"Mejores parámetros: {grid_rf.best_params_}")

In [ ]:
# Evaluar el mejor RF en test (UNA sola vez)
mejor_rf = grid_rf.best_estimator_
y_prob_mejor_rf = mejor_rf.predict_proba(X_test)[:, 1]
y_pred_mejor_rf = mejor_rf.predict(X_test)

print(f"AUC test:      {roc_auc_score(y_test, y_prob_mejor_rf):.3f}")
print(f"Recall test:   {recall_score(y_test, y_pred_mejor_rf):.3f}")
print(f"Accuracy test: {accuracy_score(y_test, y_pred_mejor_rf):.3f}")

**Interpretación:** GridSearchCV encontró la combinación óptima de hiperparámetros usando solo datos de train (vía CV). El AUC en test es nuestra evaluación final honesta: no se usó para elegir hiperparámetros.

### Ejercicio 4: GridSearchCV con Logística (12 min)

1. Define un `param_grid` con `C = [0.001, 0.01, 0.1, 1.0, 10.0]`.
2. Usa `GridSearchCV` con `LogisticRegression(max_iter=1000, class_weight="balanced", solver="saga", random_state=42)`.
3. Reporta el mejor C, la mejor AUC (CV) y la AUC en test.

In [ ]:
# Tu código aquí


---

## 10. Selección de modelo

Ahora comparamos los 3 modelos tuneados con cross-validation para elegir el ganador.

### Ejercicio 5: Seleccionar el mejor modelo (10 min)

1. Usa `cross_val_score` con `cv=5` y `scoring="roc_auc"` para los 3 modelos tuneados.
2. Imprime `mean ± std` de cada uno.
3. ¿Cuál es el mejor modelo?

In [ ]:
# Tu código aquí


---

## 11. Gradio — Demo interactiva

Vamos a crear una interfaz para que cualquier persona (médico, paciente) pueda ingresar sus datos y obtener una predicción de riesgo de diabetes.

In [ ]:
# !pip install gradio

In [ ]:
import gradio as gr

def predecir_diabetes(embarazos, glucosa, presion, pliegue, insulina, bmi, pedigri, edad):
    """Recibe datos del paciente y devuelve riesgo de diabetes."""
    paciente = pd.DataFrame([{
        "preg": embarazos,
        "plas": glucosa,
        "pres": presion,
        "skin": pliegue,
        "insu": insulina,
        "mass": bmi,
        "pedi": pedigri,
        "age": edad,
    }])

    proba = mejor_rf.predict_proba(paciente)[0, 1]

    if proba >= 0.5:
        nivel = "ALTO"
    elif proba >= 0.3:
        nivel = "MEDIO"
    else:
        nivel = "BAJO"

    return f"Riesgo de diabetes: {proba:.1%} ({nivel})"

In [ ]:
demo = gr.Interface(
    fn=predecir_diabetes,
    inputs=[
        gr.Slider(0, 17, value=1, step=1, label="Embarazos"),
        gr.Slider(50, 200, value=100, step=1, label="Glucosa (mg/dL)"),
        gr.Slider(30, 130, value=70, step=1, label="Presión arterial (mm Hg)"),
        gr.Slider(5, 60, value=20, step=1, label="Pliegue cutáneo (mm)"),
        gr.Slider(10, 850, value=80, step=1, label="Insulina (µU/ml)"),
        gr.Slider(15, 60, value=25, step=0.5, label="BMI (kg/m²)"),
        gr.Slider(0.05, 2.5, value=0.5, step=0.01, label="Función pedigrí"),
        gr.Slider(18, 80, value=30, step=1, label="Edad"),
    ],
    outputs=gr.Text(label="Resultado"),
    title="Predictor de Riesgo de Diabetes (Random Forest)",
    description="Ingresa los datos del paciente para obtener una estimación del riesgo de diabetes.",
)

demo.launch()

### Ejercicio 6: Prueba la demo (10 min)

1. Prueba con un paciente de **alto riesgo**: 45 años, glucosa 180, BMI 35.
2. Prueba con un paciente de **bajo riesgo**: 25 años, glucosa 85, BMI 22.
3. ¿Los resultados tienen sentido clínico?

In [ ]:
# Tu código aquí


---

## Resumen

| Concepto | Lo que aprendimos |
|----------|-------------------|
| **EDA con datos reales** | Los ceros en variables clínicas son datos faltantes disfrazados. Siempre explorar antes de modelar. |
| **Logística** | Modelo lineal interpretable (coeficientes). Buen baseline. |
| **Árbol de decisión** | Interpretable (reglas legibles), pero inestable y tiende a sobreajustar. |
| **Random Forest** | Muchos árboles con bagging + features aleatorias. Más estable y preciso. |
| **Interpretabilidad** | Logística: coeficientes. Árbol: plot_tree + export_text. RF: feature_importances. |
| **Confusion Matrix** | Herramienta clave para entender FN y FP en contexto clínico. |
| **n_estimators** | 100-200 árboles suele ser suficiente. Más allá la mejora es marginal. |
| **Parámetros vs Hiperparámetros** | Los parámetros los aprende el modelo; los hiperparámetros los elegimos nosotros. |
| **Cross-Validation** | Evaluar sin contaminar el test set. Dividir train en K folds y rotar. |
| **GridSearchCV** | Búsqueda exhaustiva de la mejor combinación de hiperparámetros. |
| **Selección de modelo** | Comparar con CV, elegir el mejor, evaluar en test UNA vez. |
| **Gradio** | Demo interactiva: input → predict → output. Funciona en Jupyter. |